# Analysis for the Project: Debt-Based License Suspension

## The Main Questions:
- What are the reasons for license suspensions? [Jassim]
- What % are not falling under safety related? [Rachel, Alberto]
- What % are debt based? [Sarah]
- How many people lose their license over nothing dangerous? [Yuan, Ethan]

In [35]:
import numpy as np
import polars as pl
import pandas as pd
import pathlib

In [36]:
reinstatement_path = pathlib.Path.cwd().parent.joinpath('data', r'Reinstatement_information_Part_2 (1).xlsx')
reinstatement = pd.read_excel(reinstatement_path, engine='calamine', sheet_name=0)

reinstatement['fi64SanctionKey'] = -1
reinstatement['flngAccountKey'] = -1

reinstatement = pl.from_dataframe(
    pd.concat(
        [t1 := reinstatement, 
         t2 := pd.read_excel(reinstatement_path, engine='calamine', sheet_name=1), 
         t3 := pd.read_excel(reinstatement_path, engine='calamine', sheet_name=2)
         ], 
        axis=0
    )
)

assert(reinstatement.shape[0] == t1.shape[0] + t2.shape[0] + t3.shape[0])

print(reinstatement.shape)
reinstatement.head()

(148199, 8)


Year Reinstated,Sanction Description,Term,Days Sus/Rev,Authority,Issuing Agency,fi64SanctionKey,flngAccountKey
i64,str,str,str,str,str,i64,i64
2020,"""NON-PAYMENT OF…","""Indefinite""","""1072""",null,null,-1,-1
2020,"""PROPERTY DAMAG…","""Indefinite""","""714""",null,null,-1,-1
2020,"""NRVC Withdrawa…","""Indefinite""","""1191""",null,null,-1,-1
2020,"""NRVC Withdrawa…","""Indefinite""","""924""",null,null,-1,-1
2020,"""94C35 __ PRESE…","""Definite""","""365""","""PD_GRA""","""Grafton Police…",-1,-1


In [37]:
reinstatement.null_count()

Year Reinstated,Sanction Description,Term,Days Sus/Rev,Authority,Issuing Agency,fi64SanctionKey,flngAccountKey
u32,u32,u32,u32,u32,u32,u32,u32
0,0,12,0,119770,119770,0,0


In [38]:
suspension_path = pathlib.Path.cwd().parent.joinpath('data', r'Suspension_information_Part_1 (1).xlsx')
suspension = pd.read_excel(suspension_path, engine='calamine', sheet_name=0)
suspension['fi64SanctionKey'] = -1
suspension['Sanction Type'] = ''

suspension = pl.from_dataframe(
    pd.concat(
        [t1 := suspension, 
         t2 := pd.read_excel(suspension_path, engine='calamine', sheet_name=1), 
         t3 := pd.read_excel(suspension_path, engine='calamine', sheet_name=2)
         ], 
        axis=0
    )
)

assert(suspension.shape[0] == t1.shape[0] + t2.shape[0] + t3.shape[0])

print(suspension.shape)
suspension.head()

(342783, 8)


Year,Description,Term,Days Sus/Rev,Authority,Issuing Agency,fi64SanctionKey,Sanction Type
i64,str,str,str,str,str,i64,str
2020,"""Operating Unde…","""Definite""","""2920""",null,null,-1,""""""
2020,"""Operating Unde…","""Definite""","""730""",null,null,-1,""""""
2020,"""Operating Unde…","""Definite""","""2920""","""PD_SD7""","""State Police T…",-1,""""""
2020,"""Ignition Inter…","""Indefinite""","""INDEFINITE""",null,null,-1,""""""
2020,"""Operating Unde…","""Definite""","""730""","""PD_LAK""","""Lakeville Poli…",-1,""""""


In [39]:
suspension.null_count()

Year,Description,Term,Days Sus/Rev,Authority,Issuing Agency,fi64SanctionKey,Sanction Type
u32,u32,u32,u32,u32,u32,u32,u32
0,0,6680,0,340907,340907,0,0


In [40]:
sus_reasons = suspension.unique('Description')['Description'].to_list()
print(len(sus_reasons))

157


In [41]:
cur_suspended_path = pathlib.Path.cwd().parent.joinpath('data', r'Currently_Suspended_information_Part_3_as_of_12082_022 (1).xlsx')
cur_suspended = pl.read_excel(cur_suspended_path, engine='calamine')
print(cur_suspended.shape)
cur_suspended.head()

(693100, 5)


Description,Term,Days Sus/Rev,Authority,Issuing Agency
str,str,str,null,null
"""COURT DEFAULT""","""Indefinite""","""INDEFINITE""",null,null
"""W00 - Non-ACD …","""Indefinite""","""INDEFINITE""",null,null
"""Ignition Inter…","""Indefinite""","""INDEFINITE""",null,null
"""W00 - Non-ACD …","""Indefinite""","""INDEFINITE""",null,null
"""FSU __ 5 SURCH…","""Indefinite""","""INDEFINITE""",null,null


In [42]:
cur_suspended.null_count()

Description,Term,Days Sus/Rev,Authority,Issuing Agency
u32,u32,u32,u32,u32
0,10927,0,693100,693100


In [43]:
default = cur_suspended.filter((pl.col("Description") == "PAYMENT DEFAULT") & (pl.col("Term") == "Indefinite"))
print(f'{default.shape[0]} suspensions due to payment default, {round(default.shape[0]/cur_suspended.shape[0]*100, 3)}% of {cur_suspended.shape[0]} total suspensions')

94841 suspensions due to payment default, 13.684% of 693100 total suspensions
